# PostgreSQL queries — 1NF before vs after

Use this notebook to explore why 1NF matters: the same questions are hard against
`orders_unnormalized` and straightforward against the normalized tables.

Uses the shared `scripts.db` helpers (same connection as `make seed` / `seed.ipynb`).

Run `make compose_up` and seed via `seed.ipynb` or `make seed`, then open this notebook at `http://localhost:8888`.

In [ ]:
from scripts.db import query

## Unnormalized table (violates 1NF)

Multi-valued columns: `customer_phone_numbers` and `line_items` are delimited strings.

In [ ]:
query(
    """
    SELECT id, customer_name, customer_phone_numbers, order_date, line_items
    FROM orders_unnormalized
    ORDER BY id
    LIMIT 10
    """
)

### Pain point: find orders by phone number

Without atomic phone values you must use brittle `LIKE` matching on a concatenated string
(false positives possible, no index-friendly equality).

In [ ]:
query(
    """
    SELECT id, customer_name, customer_phone_numbers
    FROM orders_unnormalized
    WHERE customer_phone_numbers LIKE %s
    LIMIT 10
    """,
    ("%555-%",),
)

### Pain point: order totals from delimited line items

SQL cannot cleanly `SUM` quantity × price without parsing strings (here we just show the raw text).

In [ ]:
query(
    """
    SELECT id, customer_name, line_items
    FROM orders_unnormalized
    ORDER BY id
    LIMIT 10
    """
)

## Normalized tables (1NF)

Atomic values: one phone per row, one line item per row, customers and orders as separate entities.

In [ ]:
query("SELECT id, name FROM customers ORDER BY id LIMIT 10")

In [ ]:
query(
    """
    SELECT id, customer_id, phone_number
    FROM customer_phone_numbers
    ORDER BY id
    LIMIT 10
    """
)

In [ ]:
query(
    """
    SELECT id, customer_id, order_date
    FROM orders
    ORDER BY id
    LIMIT 10
    """
)

In [ ]:
query(
    """
    SELECT id, order_id, product_sku, quantity, unit_price_cents
    FROM order_items
    ORDER BY id
    LIMIT 10
    """
)

### Same question, cleaner answer: find by phone number

Exact match on an atomic `phone_number` column with a proper join.

In [ ]:
query(
    """
    SELECT c.id AS customer_id, c.name, p.phone_number
    FROM customers c
    JOIN customer_phone_numbers p ON p.customer_id = c.id
    WHERE p.phone_number LIKE %s
    ORDER BY c.id
    LIMIT 10
    """,
    ("555-%",),
)

### Same question, cleaner answer: order totals

`SUM(quantity * unit_price_cents)` works because each line item is a row.

In [ ]:
query(
    """
    SELECT
      o.id AS order_id,
      c.name AS customer_name,
      o.order_date,
      SUM(oi.quantity * oi.unit_price_cents) AS total_cents
    FROM orders o
    JOIN customers c ON c.id = o.customer_id
    JOIN order_items oi ON oi.order_id = o.id
    GROUP BY o.id, c.name, o.order_date
    ORDER BY o.order_date DESC
    LIMIT 20
    """
)

## Row counts (sanity check)

Unnormalized row count equals order count after normalization;
phone numbers and line items expand into multiple rows.

In [ ]:
query(
    """
    SELECT
      (SELECT COUNT(*) FROM orders_unnormalized) AS unnormalized_orders,
      (SELECT COUNT(*) FROM customers) AS customers,
      (SELECT COUNT(*) FROM customer_phone_numbers) AS phone_numbers,
      (SELECT COUNT(*) FROM orders) AS orders,
      (SELECT COUNT(*) FROM order_items) AS order_items
    """
)